# eDRis Phase 1: APTOS Baseline Classifier (Colab)


This notebook trains a baseline ResNet18 model on the APTOS 2019 dataset to predict DR severity (Level 0-4) and calculates the critical metrics for referable DR (Level 2+).


## 1. Mount Google Drive
If your dataset is in Google Drive, run this cell and adjust the paths.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# UPDATE THESE PATHS TO WHERE YOUR DATA IS IN GOOGLE DRIVE
IMG_DIR = '/content/drive/MyDrive/eDRis/datasets/classification/aptos2019/train_images'
TRAIN_CSV = '/content/drive/MyDrive/eDRis/datasets/classification/aptos2019/splits/train_split.csv'
VAL_CSV = '/content/drive/MyDrive/eDRis/datasets/classification/aptos2019/splits/val_split.csv'



## 2. Imports and Dataset Definition


In [ ]:
import os
import time
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from torchvision import models, transforms
from PIL import Image
from sklearn.metrics import confusion_matrix, classification_report
import warnings
warnings.filterwarnings('ignore')

class APTOSDataset(Dataset):
    def __init__(self, csv_file, img_dir, transform=None):
        self.df = pd.read_csv(csv_file)
        self.img_dir = img_dir
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        img_name = os.path.join(self.img_dir, f"{row['id_code']}.png")
        image = Image.open(img_name).convert('RGB')
        label = row['diagnosis']
        
        if self.transform:
            image = self.transform(image)
            
        return image, label



## 3. Preprocessing & DataLoaders


In [ ]:
def get_transforms():
    train_transform = transforms.Compose([
        transforms.Resize((224, 224)),
        transforms.RandomHorizontalFlip(),
        transforms.RandomRotation(15),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406],
                             std=[0.229, 0.224, 0.225])
    ])
    val_transform = transforms.Compose([
        transforms.Resize((224, 224)),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406],
                             std=[0.229, 0.224, 0.225])
    ])
    return train_transform, val_transform

train_df = pd.read_csv(TRAIN_CSV)
train_tf, val_tf = get_transforms()

train_dataset = APTOSDataset(TRAIN_CSV, IMG_DIR, transform=train_tf)
val_dataset = APTOSDataset(VAL_CSV, IMG_DIR, transform=val_tf)

class_counts = train_df['diagnosis'].value_counts().sort_index().values
class_weights = 1.0 / class_counts
sample_weights = [class_weights[label] for label in train_df['diagnosis']]
sampler = WeightedRandomSampler(sample_weights, num_samples=len(sample_weights), replacement=True)

train_loader = DataLoader(train_dataset, batch_size=32, sampler=sampler, num_workers=2)
val_loader = DataLoader(val_dataset, batch_size=32, shuffle=False, num_workers=2)



## 4. Model Training


In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

model = models.resnet18(weights='DEFAULT')
num_ftrs = model.fc.in_features
model.fc = nn.Linear(num_ftrs, 5)
model = model.to(device)

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=1e-4)
num_epochs = 5 

for epoch in range(num_epochs):
    model.train()
    running_loss = 0.0
    correct = 0
    total = 0
    
    start_time = time.time()
    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)
        
        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        
        running_loss += loss.item() * images.size(0)
        _, preds = torch.max(outputs, 1)
        correct += torch.sum(preds == labels.data)
        total += labels.size(0)
        
    epoch_loss = running_loss / total
    epoch_acc = correct.double() / total
    print(f"Epoch {epoch+1}/{num_epochs} - Loss: {epoch_loss:.4f}, Acc: {epoch_acc:.4f} - Time: {time.time()-start_time:.1f}s")



## 5. Evaluation and Level 2+ Metrics


In [ ]:
model.eval()
all_preds = []
all_labels = []

with torch.no_grad():
    for images, labels in val_loader:
        images = images.to(device)
        outputs = model(images)
        _, preds = torch.max(outputs, 1)
        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(labels.numpy())
        
cm = confusion_matrix(all_labels, all_preds)
print("Confusion Matrix:")
print(cm)

print("\nClassification Report (5-class):")
print(classification_report(all_labels, all_preds))

# Calculate Level 2+ Sensitivity & Specificity
binary_labels = [1 if x >= 2 else 0 for x in all_labels]
binary_preds = [1 if x >= 2 else 0 for x in all_preds]

tn, fp, fn, tp = confusion_matrix(binary_labels, binary_preds).ravel()

sensitivity = tp / (tp + fn) if (tp + fn) > 0 else 0.0
specificity = tn / (tn + fp) if (tn + fp) > 0 else 0.0

print("\n--- Binary Referable DR (Level 2+) Metrics ---")
print(f"True Positives (TP): {tp}")
print(f"True Negatives (TN): {tn}")
print(f"False Positives (FP): {fp}")
print(f"False Negatives (FN): {fn}")
print(f"Sensitivity (Target > 0.90): {sensitivity:.4f}")
print(f"Specificity (Target > 0.85): {specificity:.4f}")

# Save the model
torch.save(model.state_dict(), '/content/drive/MyDrive/eDRis/models/aptos_resnet18_baseline.pth')
print("Model saved to Drive!")


# ---------------------------------------------------------
# NEW: Export ONNX Model & Validation Results for MATLAB
# ---------------------------------------------------------
import torch.onnx

# Export to ONNX
dummy_input = torch.randn(1, 3, 224, 224).to(device)
onnx_path = '/content/drive/MyDrive/eDRis/models/dr_resnet18.onnx'
os.makedirs('/content/drive/MyDrive/eDRis/models/', exist_ok=True)

torch.onnx.export(model, dummy_input, onnx_path, 
                  export_params=True, opset_version=11, 
                  do_constant_folding=True, 
                  input_names = ['input'], output_names = ['output'], 
                  dynamic_axes={'input' : {0 : 'batch_size'}, 'output' : {0 : 'batch_size'}})
print(f'ONNX Model exported to {onnx_path}')

# Generate Probabilities for ROC Curve
import torch.nn.functional as F

model.eval()
all_probs = []
all_true = []

with torch.no_grad():
    for images, labels in val_loader:
        images = images.to(device)
        outputs = model(images)
        probs = F.softmax(outputs, dim=1)
        
        # Probability of being Level 2, 3, or 4
        referable_probs = torch.sum(probs[:, 2:], dim=1).cpu().numpy()
        all_probs.extend(referable_probs)
        all_true.extend(labels.numpy())

# Save to CSV for MATLAB ROC Script
df_results = pd.DataFrame({
    'True_Label': all_true,
    'Predicted_Prob_Level_2_Plus': all_probs
})
csv_path = '/content/drive/MyDrive/eDRis/results/validation_results.csv'
os.makedirs('/content/drive/MyDrive/eDRis/results/', exist_ok=True)
df_results.to_csv(csv_path, index=False)
print(f'Validation probabilities saved to {csv_path}')
